In [1]:
"""
Cutting Stock Problem - Column Generation (Winston & Wayne, Example 2, p.570)
=============================================================================
Woodco sells 3-ft, 5-ft, and 9-ft lumber pieces.
Demand: 25 three-ft boards, 20 five-ft boards, 15 nine-ft boards.
Raw boards are 17-ft. Minimize total waste (= minimize total boards cut).

Uses scipy.optimize:
  - linprog  → master LP (continuous relaxation)
  - milp     → knapsack pricing subproblem (integer)

Column generation loop:
  1. Solve restricted master LP → get dual (shadow) prices
  2. Solve knapsack subproblem:  max  duals · a - 1
                                 s.t. 3a3 + 5a5 + 9a9 <= 17,  a integer >= 0
  3. If knapsack z > 0: add new pattern, go to 1
     Else: optimal found
"""

import numpy as np
from scipy.optimize import linprog, milp, LinearConstraint, Bounds

# ─────────────────────────────────────────────────────────────────────────────
# Problem data
# ─────────────────────────────────────────────────────────────────────────────
BOARD_LENGTH  = 17
PIECE_LENGTHS = np.array([3, 5, 9], dtype=int)
DEMANDS       = np.array([25, 20, 15], dtype=int)
N             = len(PIECE_LENGTHS)

def pattern_label(p):
    return f"[3ft={p[0]}, 5ft={p[1]}, 9ft={p[2]}]"

# ─────────────────────────────────────────────────────────────────────────────
# Solve master LP (continuous relaxation)
#
# min  sum(x_j)
# s.t. A @ x >= demands      (each demand must be met)
#      x >= 0
#
# scipy linprog uses A_ub @ x <= b_ub, so we negate: -A @ x <= -demands
# Duals (marginals) for -A@x <= -d  satisfy:  shadow_price = -marginal
# ─────────────────────────────────────────────────────────────────────────────
def solve_master(patterns):
    n_pat = len(patterns)
    A     = np.array(patterns, dtype=float).T   # shape (N, n_pat)

    c     = np.ones(n_pat)
    A_ub  = -A
    b_ub  = -DEMANDS.astype(float)
    bds   = [(0, None)] * n_pat

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bds, method="highs")

    if res.status != 0:
        raise RuntimeError(f"Master LP failed: {res.message}")

    # HiGHS marginals for A_ub @ x <= b_ub are <= 0 at optimum for a min problem.
    # Our original >= constraint flips sign, so shadow price = -marginal.
    shadow_prices = -res.ineqlin.marginals

    return shadow_prices, res.x, res.fun

# ─────────────────────────────────────────────────────────────────────────────
# Solve knapsack pricing subproblem (MILP)
#
# max  duals · a - 1
# s.t. PIECE_LENGTHS · a <= BOARD_LENGTH
#      a >= 0, integer
#
# scipy milp minimises, so minimise -(duals · a) + 1, then reduced_cost = -obj-1+1
# ─────────────────────────────────────────────────────────────────────────────
def solve_knapsack(shadow_prices):
    c          = -shadow_prices.astype(float)          # minimise = maximise duals·a
    constraint = LinearConstraint(
        PIECE_LENGTHS.reshape(1, -1).astype(float),
        lb=-np.inf,
        ub=float(BOARD_LENGTH)
    )
    integrality = np.ones(N)
    bounds      = Bounds(lb=np.zeros(N), ub=np.full(N, float(BOARD_LENGTH)))

    res = milp(c, constraints=constraint, integrality=integrality, bounds=bounds)

    if res.status != 0:
        raise RuntimeError(f"Knapsack failed: {res.message}")

    new_pattern  = [int(round(v)) for v in res.x]
    reduced_cost = float(np.dot(shadow_prices, res.x) - 1)

    return reduced_cost, new_pattern

# ─────────────────────────────────────────────────────────────────────────────
# Main column generation loop
# ─────────────────────────────────────────────────────────────────────────────
def column_generation():
    # Start with Table 1's 6 patterns + artificial combo 7 [0,0,1]
    # (combo 7 gives a clean starting BFS for the 9-ft row, as in the book)
    patterns = [
        [5, 0, 0],   # combo 1
        [4, 1, 0],   # combo 2
        [2, 2, 0],   # combo 3
        [2, 0, 1],   # combo 4
        [1, 1, 1],   # combo 5
        [0, 3, 0],   # combo 6
        [0, 0, 1],   # combo 7 (artificial)
    ]

    print("=" * 65)
    print("  WOODCO CUTTING STOCK — Column Generation (Winston §10.3)")
    print("=" * 65)
    print(f"  Board length : {BOARD_LENGTH} ft")
    print(f"  Piece sizes  : {[int(x) for x in PIECE_LENGTHS]} ft")
    print(f"  Demands      : {[int(x) for x in DEMANDS]}")
    print(f"  Initial patterns: {len(patterns)}")
    print("=" * 65)

    iteration = 0

    while True:
        iteration += 1

        # ── Step 1: solve restricted master LP ──────────────────────────────
        shadow_prices, x_vals, obj = solve_master(patterns)

        print(f"\n{'─'*65}")
        print(f"  Iteration {iteration}")
        print(f"{'─'*65}")
        print(f"  Shadow prices (π) : {[round(float(p), 6) for p in shadow_prices]}")
        print(f"  Master LP obj (z) : {round(obj, 6)}")
        print(f"  Active variables  :")
        for j, (p, v) in enumerate(zip(patterns, x_vals)):
            if v > 1e-8:
                print(f"    x{j+1} = {round(v,6):>10}   {pattern_label(p)}")

        # ── Step 2: solve pricing subproblem ────────────────────────────────
        reduced_cost, new_pattern = solve_knapsack(shadow_prices)

        print(f"\n  Pricing subproblem:")
        print(f"    Best new pattern : {pattern_label(new_pattern)}")
        print(f"    Reduced cost     : {round(reduced_cost, 6)}")

        # ── Step 3: optimality check ─────────────────────────────────────────
        if reduced_cost <= 1e-6:
            print(f"\n  ✓ Reduced cost ≤ 0 → No improving column exists.")
            print(f"    LP OPTIMAL after {iteration} iteration(s).")
            break

        if new_pattern in patterns:
            print(f"  ✗ Pattern already present — stopping.")
            break

        print(f"  → Adding {pattern_label(new_pattern)} to master LP.")
        patterns.append(new_pattern)

    # ── Final solution report ─────────────────────────────────────────────────
    shadow_prices, x_vals, obj = solve_master(patterns)

    print(f"\n{'═'*65}")
    print(f"  FINAL OPTIMAL LP SOLUTION")
    print(f"{'═'*65}")
    print(f"  Objective (boards cut, LP relaxation): {round(obj, 6)}")
    print()
    print(f"  {'Var':<6} {'Value':>10}   Pattern")
    print(f"  {'───':<6} {'─────':>10}   ─────────────────────────")
    for j, (p, v) in enumerate(zip(patterns, x_vals)):
        if v > 1e-8:
            print(f"  x{j+1:<5} {round(v, 6):>10}   {pattern_label(p)}")

    # Integer solution by rounding up
    x_int        = [int(np.ceil(v)) if v > 1e-8 else 0 for v in x_vals]
    total_boards = sum(x_int)

    print(f"\n  Rounded-up integer solution (total boards = {total_boards}):")
    for j, (p, v) in enumerate(zip(patterns, x_int)):
        if v > 0:
            print(f"    x{j+1} = {v}   {pattern_label(p)}")

    print(f"\n  Demand satisfaction check:")
    for i, (size, demand) in enumerate(zip(PIECE_LENGTHS, DEMANDS)):
        supplied = sum(x_int[j] * patterns[j][i] for j in range(len(patterns)))
        status   = "✓" if supplied >= demand else "✗"
        print(f"    {size}-ft boards: need {demand}, get {supplied}  {status}")

    total_demand_ft = int(np.dot(DEMANDS, PIECE_LENGTHS))
    total_cut_ft    = total_boards * BOARD_LENGTH
    waste_ft        = total_cut_ft - total_demand_ft
    print(f"\n  Total raw boards used : {total_boards}")
    print(f"  Total wood cut (ft)   : {total_cut_ft}")
    print(f"  Total demand (ft)     : {total_demand_ft}")
    print(f"  Total waste (ft)      : {waste_ft}")
    print("=" * 65)

# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    column_generation()

  WOODCO CUTTING STOCK — Column Generation (Winston §10.3)
  Board length : 17 ft
  Piece sizes  : [3, 5, 9] ft
  Demands      : [25, 20, 15]
  Initial patterns: 7

─────────────────────────────────────────────────────────────────
  Iteration 1
─────────────────────────────────────────────────────────────────
  Shadow prices (π) : [0.166667, 0.333333, 0.5]
  Master LP obj (z) : 18.333333
  Active variables  :
    x2 =        2.5   [3ft=4, 5ft=1, 9ft=0]
    x5 =       15.0   [3ft=1, 5ft=1, 9ft=1]
    x6 =   0.833333   [3ft=0, 5ft=3, 9ft=0]

  Pricing subproblem:
    Best new pattern : [3ft=0, 5ft=3, 9ft=0]
    Reduced cost     : 0.0

  ✓ Reduced cost ≤ 0 → No improving column exists.
    LP OPTIMAL after 1 iteration(s).

═════════════════════════════════════════════════════════════════
  FINAL OPTIMAL LP SOLUTION
═════════════════════════════════════════════════════════════════
  Objective (boards cut, LP relaxation): 18.333333

  Var         Value   Pattern
  ───         ─────   ──────